# MotorGuard SENATI | Cuaderno Jupyter
**Autora:** Lorena Abigail Coca Ccorpa  
**Institución:** SENATI  
**Fecha:** noviembre de 2026  
**Tema:** Machine Learning no supervisado: detección de anomalías en motores

![Logo SENATI](../static/logo.svg)

## 1. Objetivo
Analizar variables simuladas de motores y detectar observaciones poco habituales mediante Isolation Forest. Este modelo no identifica fallas reales.

## 2. Metodología
1. Generación de 318 registros sintéticos (300 habituales y 18 inusuales).
2. Exploración y limpieza: variables numéricas, ausencia de etiquetas para entrenamiento.
3. Escalamiento StandardScaler, entrenamiento IsolationForest.
4. Visualización, interpretación y demostración de predicciones.

**Nota:** la proporción de anomalías fijada en el algoritmo no es una tasa de fallas del mundo real.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
ROOT = Path.cwd()
if not (ROOT / 'app.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from app import generate_data, FEATURES
print('Ruta:', ROOT)

## 3. Datos de entrenamiento y estadística descriptiva

In [ ]:
df = generate_data(seed=42)
print('Dimensiones:',df.shape)
display(df.head())
display(df.describe().round(2))
print('Nulos:',df.isna().sum().to_dict())

## 4. Gráficos exploratorios

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(10,7))
for ax, col in zip(axes.ravel(),FEATURES):
    ax.hist(df[col],bins=24,color='#1587a8',edgecolor='white')
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 5. Modelo no supervisado
Se entrena sin etiqueta de fallas. Se utiliza escalamiento para mantener un pipeline coherente y reproducible.

In [ ]:
pipeline = make_pipeline(StandardScaler(), IsolationForest(n_estimators=200,contamination=0.06,random_state=42))
pipeline.fit(df[FEATURES])
labels = pipeline.predict(df[FEATURES])
scores = pipeline.decision_function(df[FEATURES])
df['anomalia'] = labels == -1
df['puntaje'] = scores
print('Registros señalados:',int(df['anomalia'].sum()))
display(df[df.anomalia].head())

## 6. Interpretación visual

In [ ]:
plt.figure(figsize=(9,5))
plt.scatter(df.loc[~df.anomalia,'rpm'],df.loc[~df.anomalia,'temperatura_c'],s=15,label='Habitual',alpha=.6)
plt.scatter(df.loc[df.anomalia,'rpm'],df.loc[df.anomalia,'temperatura_c'],s=40,label='Poco habitual',marker='x')
plt.xlabel('RPM'); plt.ylabel('Temperatura (°C)');plt.legend();plt.title('Señalización estadística');plt.show()

## 7. Predicción para un motor de ejemplo

In [ ]:
ejemplo = pd.DataFrame([{'rpm':1450,'temperatura_c':55,'vibracion_mm_s':2.0,'corriente_a':6.5}])
pred = pipeline.predict(ejemplo)[0]
print('Clasificación:', 'Poco habitual' if pred == -1 else 'Habitual')
print('Índice:', round(float(pipeline.decision_function(ejemplo)[0]),5))

## 8. Pruebas y conclusiones
La aplicación valida los rangos de entrada, expone `/health` y devuelve JSON en `/api/predict`.

**Conclusión:** Isolation Forest detecta observaciones aisladas respecto del conjunto simulado; requiere datos de sensores reales, evaluación por expertos y validación externa antes de usarlo para mantenimiento real.

**Bibliografía:** Liu, Ting y Zhou (2008), *Isolation Forest*, ICDM. Scikit-learn, documentación de IsolationForest. Flask, documentación oficial.

**Sitio web:** enlazar aquí la dirección de Render una vez publicado.